In [1]:
poi = 'Pegula'
import uuid

In [2]:
def magnitude(x):
    if x:
        return (x['x']**2 + x['y']**2 + x['z']**2)**0.5*3.6
    else:
        return None

In [3]:
# CV PROCESSING
import json
import pandas as pd
import numpy as np
import requests
import os

point_number_dict = {
    '0': 0,
    '15': 1,
    '30': 2,
    '40': 3,
    'Ad': 4
}

def define_point_number(x, y):
    return point_number_dict[x] + point_number_dict[y] + 1

In [4]:
import numpy as np
try:
    from scipy.signal import savgol_filter
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False

# ---- per-frame movement extraction (from cv_data['players_positions']) ------ #
MOVE_SMOOTH_WIN = 9          # Savitzky-Golay window (frames) for position/speed smoothing (tune me)
MOVE_MIN_FRAMES = 5          # need at least this many tracked frames in a run
MOVE_DT_DEFAULT = 0.04       # seconds/frame (25 fps); overridden from the data when possible


def _ts_seconds(ts):
    try:
        h, m, s = str(ts).split(":")
        return float(h) * 3600 + float(m) * 60 + float(s)
    except Exception:
        return None


def _estimate_dt(positions):
    """Seconds per frame from the players_positions timestamps (fallback 0.04)."""
    valid = [r for r in positions if r and r.get("time") and r["time"].get("frame") is not None]
    try:
        a, b = valid[0], valid[min(200, len(valid) - 1)]
        fa, fb = a["time"]["frame"], b["time"]["frame"]
        ta, tb = _ts_seconds(a["time"]["timestamp"]), _ts_seconds(b["time"]["timestamp"])
        if fb > fa and ta is not None and tb is not None and tb > ta:
            return (tb - ta) / (fb - fa)
    except Exception:
        pass
    return MOVE_DT_DEFAULT


def _sg(a, win, deriv, dt):
    """Smoothed derivative: Savitzky-Golay if available, else finite differences."""
    a = np.asarray(a, float)
    n = len(a)
    if not _HAVE_SCIPY or n < 5:
        if deriv == 0:
            return a
        g = np.gradient(a, dt)
        return g if deriv == 1 else np.gradient(g, dt)
    w = min(win, n)
    w = w if w % 2 == 1 else w - 1
    if w < 5:
        w = 5
    return savgol_filter(a, w, 2, deriv=deriv, delta=dt)


def _clean_series(vals):
    """List possibly containing None -> float array with interior gaps interpolated."""
    a = np.array([np.nan if v is None else v for v in vals], float)
    good = ~np.isnan(a)
    if good.sum() < 3:
        return None
    return np.interp(np.arange(len(a)), np.flatnonzero(good), a[good])


def _point_movement(point, positions, f2i, dt):
    """For each shot, the HITTER's movement while REACHING the ball: their
    per-frame position track over [previous shot's hit frame, this shot's hit
    frame] (opponent hit -> player reaches ball), smoothed then differentiated.
    Returns a list aligned to point['shots'] (None where not computable). Metric
    units: speed m/s, acceleration m/s^2 (deceleration = the min, i.e. negative)."""
    shots = point.get("shots", [])
    out = [None] * len(shots)
    if len(shots) < 2 or not positions:
        return out
    f0, f1 = shots[0]["time"]["frame"], shots[-1]["time"]["frame"]
    frames = [f for f in range(f0, f1 + 1) if f in f2i]
    if len(frames) < MOVE_MIN_FRAMES:
        return out
    rows = [positions[f2i[f]] for f in frames]
    fidx = {f: i for i, f in enumerate(frames)}
    # smoothed velocity of each tracked player over the whole point (better
    # derivative behaviour at window edges than smoothing each window alone)
    vel = {}
    for key in ("player1", "player2"):
        x = _clean_series([(r[key]["x"] if r.get(key) else None) for r in rows])
        y = _clean_series([(r[key]["y"] if r.get(key) else None) for r in rows])
        vel[key] = None if (x is None or y is None) else (
            _sg(x, MOVE_SMOOTH_WIN, 1, dt), _sg(y, MOVE_SMOOTH_WIN, 1, dt), x, y)
    for k in range(1, len(shots)):
        s, prev = shots[k], shots[k - 1]
        loc = s.get("player_location")
        fk, fk1 = s["time"]["frame"], prev["time"]["frame"]
        if loc is None or fk not in fidx or fk1 not in fidx:
            continue
        a, b = fidx[fk1], fidx[fk]
        if b - a < 3:
            continue
        rk = rows[b]

        def _d(key):
            return ((rk[key]["x"] - loc["x"]) ** 2 + (rk[key]["y"] - loc["y"]) ** 2) if rk.get(key) else 1e18

        key = "player1" if _d("player1") < _d("player2") else "player2"   # which track is the hitter
        if vel[key] is None:
            continue
        vx, vy, px, py = vel[key]
        sx, sy = vx[a:b + 1], vy[a:b + 1]
        speed = np.hypot(sx, sy)
        lon = np.abs(sx)                       # longitudinal (x) speed magnitude
        lat = np.abs(sy)                       # lateral (y) speed magnitude
        f_acc = _sg(speed, MOVE_SMOOTH_WIN, 1, dt)   # d(speed)/dt: >0 accel, <0 decel
        l_acc = _sg(lon, MOVE_SMOOTH_WIN, 1, dt)
        t_acc = _sg(lat, MOVE_SMOOTH_WIN, 1, dt)
        T = (b - a) * dt
        se = float(np.hypot(px[b] - px[a], py[b] - py[a]) / T) if T > 0 else None
        out[k] = {
            "move_n": int(b - a + 1),
            "move_spd_avg": float(speed.mean()), "move_spd_max": float(speed.max()),
            "move_spd_min": float(speed.min()), "move_spd_se": se,
            "move_acc_max": float(f_acc.max()), "move_dec_max": float(f_acc.min()),
            "move_lon_avg": float(lon.mean()), "move_lon_max": float(lon.max()), "move_lon_min": float(lon.min()),
            "move_lon_acc_max": float(l_acc.max()), "move_lon_dec_max": float(l_acc.min()),
            "move_lat_avg": float(lat.mean()), "move_lat_max": float(lat.max()), "move_lat_min": float(lat.min()),
            "move_lat_acc_max": float(t_acc.max()), "move_lat_dec_max": float(t_acc.min()),
        }
    return out


def cv_json_to_df(cv_data):
    if int(cv_data['version']['major']) < 4:
        print('Version must be higher >= 4')
        raise Exception
    #with open('cv_data_EX.json', 'w') as f:
    #    json.dump(cv_data, f, indent=4)
    camera_info = cv_data['camera_info']
    metadata = cv_data['metadata']
    data = []
    version_cv = cv_data['version']['major'] + '.' + cv_data['version']['minor'] + '.' + cv_data['version']['patch']
    # per-frame player positions for movement (accel/decel/min-max, lon/lat)
    _positions = cv_data.get('players_positions') or []
    _f2i = {r['time']['frame']: i for i, r in enumerate(_positions)
            if r and r.get('time') and r['time'].get('frame') is not None}
    _dt = _estimate_dt(_positions) if _f2i else MOVE_DT_DEFAULT
    print('DATA', len(cv_data['points']))

    for i, point in enumerate(cv_data['points']):

        shot_number = 1
        movement = _point_movement(point, _positions, _f2i, _dt)
        first_serve_time = point['first_serve_time']
        second_serve_time = point['second_serve_time']
        set_number = int(point['score']['p1_set']) + int(point['score']['p2_set']) + 1
        game_number = int(point['score']['p1_game']) + int(point['score']['p2_game']) + 1
        if game_number == 13:
            try:
                point_number = int(point['score']['p1_point']) + int(point['score']['p2_point']) + 1
            except:
                point_number = 0
        else:
            try:
                point_number = define_point_number(point['score']['p1_point'], point['score']['p2_point'])
            except:
                point_number = 0
                continue
        
        for _si, shot in enumerate(point['shots']):
            if shot['type'] == 'serve':
                shot_number = 1
            elif shot['type'] == 'return':
                shot_number = 2
            #elif shot['type'] == 'groundstroke' and shot_number < 3:
            #    shot_number = 3
            h, m, s = shot['time']['timestamp'].split(':')
            seconds = float(h)*3600 + float(m.lstrip(''))*60 + float(s)
            outcome = 'unknown'
            if 'outcome' in shot:
                outcome = shot['outcome']
            trajectory_3d_column = 'trajectory_3d'
            if 'trajectory_3d_beta' in shot:
                trajectory_3d_column = 'trajectory_3d_beta'
            zs = []
            if shot['toss_trajectory_3d'] and shot['toss_trajectory_3d'].get('positions'):
                for position in shot['toss_trajectory_3d'].get('positions'):
                    zs.append(position.get('position').get('z'))
            if shot['bounce_location']:
                data.append({
                    'z_toss': zs,
                    'match_id': cv_data['match_id'],
                    'server': point['server'],
                    'point_winner': point['point_winner'],
                    'top_player': point['top_player'],
                    'start_time': point['start_time']['timestamp'],
                    'end_time': point['end_time']['timestamp'],
                    'Hit_frame':  shot['time']['frame'],
                    'Bounce_frame': shot['bounce_time']['frame'],
                    'Hit_time': shot['time']['timestamp'],
                    'Bounce_time': shot['bounce_time']['timestamp'],
                    'bounce_x': shot['bounce_location']['x'],
                    'bounce_y': shot['bounce_location']['y'],
                    'hit_x': shot['player_location']['x'] if shot['player_location'] else None,
                    'hit_y': shot['player_location']['y'] if shot['player_location'] else None,
                    'Player1_score_points': point['score']['p1_point'],
                    'Player2_score_points': point['score']['p2_point'],
                    'Player1_score_set': point['score']['p1_set'],
                    'Player2_score_set': point['score']['p2_set'],
                    'Player1_score_game': point['score']['p1_game'],
                    'Player2_score_game': point['score']['p2_game'],
                    'player_location_x': shot['player_location']['x'] if shot['player_location'] else None,
                    'player_location_y': shot['player_location']['y'] if shot['player_location'] else None,
                    'receiver_location_x': shot['receiving_player_location']['x'] if shot['receiving_player_location'] else None,
                    'receiver_location_y': shot['receiving_player_location']['y'] if shot['receiving_player_location'] else None,
                    'set': set_number,
                    'game': game_number,
                    'point': point_number,
                    'shot': shot_number,
                    'events.shot': shot['type'],
                    'placement_gsa': shot['placement_gsa'],
                    'stroke': shot['stroke'],
                    'serve_type': shot.get('serve_subtype'),
                    'outcome': outcome,
                    'SPEED': shot['speed'],
                    'SPEED_3D': shot['speed'] if not shot.get(trajectory_3d_column) or 'speed' not in shot[trajectory_3d_column] else magnitude(shot[trajectory_3d_column]['speed']),
                    'player_jump_height': shot.get('player_jump_height', 0) / 10.0,
                    'CONTACT_Z': shot[trajectory_3d_column]['positions'][0]['position']['z'] if trajectory_3d_column in shot and shot[trajectory_3d_column] and shot[trajectory_3d_column]['positions']  else shot['hit_location'].get('z', None) if shot.get('hit_location') else None,
                    'ON_NET_Z': shot[trajectory_3d_column]['crossing_location'].get('z') if trajectory_3d_column in shot and  shot[trajectory_3d_column] and 'crossing_location' in shot[trajectory_3d_column] and shot[trajectory_3d_column]['crossing_location'] else shot['crossing_location']['z'] if shot.get('crossing_location') else None,
                    'ON_NET_Y': shot[trajectory_3d_column]['crossing_location'].get('y') if trajectory_3d_column in shot and  shot[trajectory_3d_column] and 'crossing_location' in shot[trajectory_3d_column] and shot[trajectory_3d_column]['crossing_location'] else shot['crossing_location']['y'] if shot.get('crossing_location') else None,
                    'version_cv': version_cv,
                    'camera_info': camera_info,
                    'cv_metadata': metadata,
                    'first_serve_time': first_serve_time,
                    'second_serve_time': second_serve_time,
                })
            else:
                data.append({
                    'z_toss': zs,
                    'match_id': cv_data['match_id'],
                    'server': point['server'],
                    'point_winner': point['point_winner'],
                    'top_player': point['top_player'],
                    'distance_bottom_player': point['distance_bottom_player'],
                    'distance_top_player': point['distance_top_player'],
                    'start_time': point['start_time']['timestamp'],
                    'end_time': point['end_time']['timestamp'],
                    'Hit_frame':  shot['time']['frame'],
                    'Bounce_frame': None,
                    'Hit_time': shot['time']['timestamp'],
                    'Bounce_time': None,
                    'bounce_x': None,
                    'bounce_y': None,
                    'hit_x': shot['player_location']['x'] if shot['player_location'] else None,
                    'hit_y': shot['player_location']['y'] if shot['player_location'] else None,
                    'Player1_score_points': point['score']['p1_point'],
                    'Player2_score_points': point['score']['p2_point'],
                    'Player1_score_set': point['score']['p1_set'],
                    'Player2_score_set': point['score']['p2_set'],
                    'Player1_score_game': point['score']['p1_game'],
                    'Player2_score_game': point['score']['p2_game'],
                    'player_location_x': shot['player_location']['x'] if shot['player_location'] else None,
                    'player_location_y': shot['player_location']['y'] if shot['player_location'] else None,
                    'receiver_location_x': shot['receiving_player_location']['x'] if shot['receiving_player_location'] else None,
                    'receiver_location_y': shot['receiving_player_location']['y'] if shot['receiving_player_location'] else None,
                    'set': set_number,
                    'game': game_number,
                    'point': point_number,
                    'shot': shot_number,
                    'events.shot': shot['type'],
                    'placement_gsa': shot['placement_gsa'],
                    'stroke': shot['stroke'],
                    'serve_type': shot.get('serve_subtype'),
                    'outcome': outcome,
                    'SPEED': shot['speed'],
                    'SPEED_3D': shot['speed'] if not shot.get(trajectory_3d_column) or 'speed' not in shot[trajectory_3d_column] else magnitude(shot[trajectory_3d_column]['speed']),
                    'player_jump_height': shot.get('player_jump_height', 0) / 10.0,
                    'CONTACT_Z': shot[trajectory_3d_column]['positions'][0]['position']['z'] if trajectory_3d_column in shot and shot[trajectory_3d_column] and shot[trajectory_3d_column]['positions']  else shot['hit_location'].get('z', None) if shot.get('hit_location') else None,
                    'ON_NET_Z': shot[trajectory_3d_column]['crossing_location'].get('z') if trajectory_3d_column in shot and  shot[trajectory_3d_column] and 'crossing_location' in shot[trajectory_3d_column] and shot[trajectory_3d_column]['crossing_location'] else shot['crossing_location']['z'] if shot.get('crossing_location') else None,
                    'ON_NET_Y': shot[trajectory_3d_column]['crossing_location'].get('y') if trajectory_3d_column in shot and  shot[trajectory_3d_column] and 'crossing_location' in shot[trajectory_3d_column] and shot[trajectory_3d_column]['crossing_location'] else shot['crossing_location']['y'] if shot.get('crossing_location') else None,
                    'version_cv': version_cv,
                    'camera_info': camera_info,
                    'cv_metadata': metadata,
                    'first_serve_time': first_serve_time,
                    'second_serve_time': second_serve_time,
                })
            if _si < len(movement) and movement[_si]:
                data[-1].update(movement[_si])
            shot_number = shot_number + 1
            
    
    cvdf = pd.DataFrame(data)
    print('DATA', len(data))
    player1 = cv_data['metadata']['player1']
    player2 = cv_data['metadata']['player2']
    cvdf['returner'] = np.where(cvdf.server == player1, player2, player1)
    cvdf['bottom_player'] = np.where(cvdf.top_player == player1, player2, player1)
    cvdf['impact_player'] = np.where(cvdf.shot % 2 == 1, cvdf['server'], cvdf['returner'])
    cvdf['opponent'] = np.where(cvdf.impact_player == player1, player2, player1)
    cvdf['serve_shot'] = np.where(cvdf.shot == 1, 1, 0)
    cvdf['serve'] = cvdf.groupby(['start_time', 'Player1_score_points', 'Player2_score_points',
           'Player1_score_set', 'Player2_score_set', 'Player1_score_game',
           'Player2_score_game'])['serve_shot'].cumsum()
    cvdf['rally'] = cvdf.groupby(['start_time', 'Player1_score_points', 'Player2_score_points',
           'Player1_score_set', 'Player2_score_set', 'Player1_score_game',
           'Player2_score_game'])['serve_shot'].cumsum()
    cvdf.loc[(cvdf.serve >= 3), 'serve'] = 2

    cvdf['point_rank'] = cvdf.groupby(['set', 'game', 'point'])['start_time'].rank(method='dense')
    cvdf['point_new'] = cvdf['point'] + 2*(cvdf['point_rank']-1)
    cvdf['point'] = (cvdf['point'] + 2*(cvdf['point_rank']-1)).astype(int)
    cvdf['Bounce_frame'] = cvdf['Bounce_frame'].astype('Int64')
    cvdf[(cvdf.point != cvdf.point_new)][['set', 'game', 'point', 'point_new']].drop_duplicates()
    cvdf['placement_gsa_previous'] = cvdf.groupby(['set', 'game', 'point'])['placement_gsa'].shift(1)
    cvdf['serve_previous'] = cvdf.groupby(['set', 'game', 'point'])['serve'].shift(1)
    cvdf['events.shot_previous'] = cvdf.groupby(['set', 'game', 'point'])['events.shot'].shift(1)
    cvdf.loc[(cvdf.serve_previous == 1) & (cvdf.placement_gsa_previous != 'Out') & (cvdf.serve == 2) & (cvdf['events.shot_previous'] == 'serve') & (cvdf['events.shot'] == 'serve'), 'serve'] = 1
    cvdf[['impact_player', 'bounce_x', 'bounce_y', 'start_time', 'end_time', 'Hit_frame', 'Bounce_frame', 'Hit_time', 'Bounce_time', 'Player1_score_points', 'Player2_score_points',
       'Player1_score_set', 'Player2_score_set', 'Player1_score_game',
       'Player2_score_game', 'opponent', 'events.shot', 'shot', 'set', 'game', 'point', 'serve', 'rally']]
    cvdf['serve_number'] = cvdf.groupby(['set', 'game', 'point']).serve.transform(max)
    cvdf['rally_length'] = cvdf.groupby(['set', 'game', 'point', 'serve']).shot.transform(max)
    cvdf['last_shot'] = cvdf.groupby(['set', 'game', 'point', 'serve']).Hit_time.transform('last')
    df = cvdf
    df['player_location_y_mirrored'] = np.where(df['player_location_x'] > 0, df['player_location_y']*-1, df['player_location_y']) #dft['CONTACT_Y_mirrored'] = np.where(dft['CONTACT_X'] > 0, dft['CONTACT_Y']*-1, dft['CONTACT_Y'])
    df['video_id'] = df['match_id']
    
    alldeucead = ['1500', '1515', '3015', '4015', '4030', '0000', '0015', '0030',
       '1530', '3030', '3000', '4040', '40Ad', 'Ad40', '1540', '3040',
       '4000', '0040']
    deuce = ['1515', '4015',  '0000',  '0030',
            '3030', '3000', '4040', '1540', 
           ]
    ad = [x for x in alldeucead if x not in deuce]
    deuce_pressure_break = ['3030', '4040', '4015']
    ad_pressure_break = ['4000', '4030', 'Ad40', '40A', 'AD40', '3015']
    ad_pressure_break_reversed = ['0040', '3040', '40Ad', '1530']
    deuce_pressure_break_reversed = ['3030', '4040', '1540']
    df['gamescore'] = df['Player1_score_points'].astype(str).apply(lambda x: '00' if x == '0' else x) + df['Player2_score_points'].astype(str).apply(lambda x: '00' if x == '0' else x)
    df['deuce_or_ad'] = np.where(df.gamescore.isin(deuce), 'deuce', 'ad')
    df['placement_gsa_original'] = df['placement_gsa']

    def replace_to_zone(x):
        if x == 'A':
            return 'D'
        elif x == 'B':
            return 'C'
        elif x == 'C':
            return 'B'
        elif x == 'D':
            return 'A'
        else:
            return x
    df['placement_gsa'] = df['placement_gsa'].apply(lambda x:  x.split('-')[0] + '-' + replace_to_zone(x.split('-')[1]) if x and len(x.split('-')) == 2 else x)

    return df

In [5]:
poi

'Pegula'

In [6]:
import requests
import json

url = "https://api.goldensetanalytics.com/login/token"

payload = json.dumps({
  "userName": "sw",
  "password": "K4!Yf*%NY)Kv6@y*"
})
headers_new = {
  'Content-Type': 'application/json'
}

response = requests.request("POST", url, headers=headers_new, data=payload)

token = response.text



import requests
import json
url2 = f"https://api.goldensetanalytics.com/videos/search?searchText={poi}"
payload = {}
headers_new = {
  'Authorization': f'Bearer {token}'
}
response = requests.request("GET", url2, headers=headers_new, data=payload)
data = json.loads(response.text)

In [7]:
match_info = [' - '.join(d['title'].split(' - ')[0:2]) for d in data if poi.lower() in d['title'].lower()]# + ['Vidmanova vs Bartunkova - WTA 125 Samsun']

In [13]:
match_info = match_info[:2]

In [14]:
data = [d for d in data if ' - '.join(d['title'].split(' - ')[0:2]) in match_info]#[:3]

In [15]:
len(data)

2

In [16]:
[d['id'] for d in data]

['a6daeb26-4e09-4943-94a0-cc1c6872d6cd',
 '847f461a-9c5a-4e17-b918-baf2a524d679']

In [ ]:
# CV DOWNLOAD
import requests
import logging
import tqdm
import pathlib
import json
import pandas as pd

logger = logging.getLogger(__name__)

BEARER_TOKEN = "eyJraWQiOiJFUzZSLVJIcndSTXhPUURqSG5mdTRYaUh6M1FyWVRjNkFJUHdzcWNjVl9FIiwiYWxnIjoiUlMyNTYifQ.eyJ2ZXIiOjEsImp0aSI6IkFULldIai01eUZLTWk0eGNjTmR0R1IwbThzWEJMTUFCOXhaZV95Qko2UEFZa3MiLCJpc3MiOiJodHRwczovL2F1dGgtaW50ZXJuYWwuZ29sZGVuc2V0YW5hbHl0aWNzLmNvbS9vYXV0aDIvYXVza3k2N2drZWhkcWlhajI0eDciLCJhdWQiOiJHU0EgT3BlcmF0aW9ucyIsImlhdCI6MTcyOTI1NTU1NywiZXhwIjoxNzI5MjY5OTU3LCJjaWQiOiIwb2FnNWxyamFmWWZMQkpPZjR4NyIsInVpZCI6IjAwdWc5NmlmcWZUY3o3ZHNBNHg3Iiwic2NwIjpbInByb2ZpbGUiLCJlbWFpbCIsInJvbGVzIiwib3BlbmlkIl0sImF1dGhfdGltZSI6MTcyOTAyODM5NSwic3ViIjoiYXhlbC52bGFtaW5jayIsInJvbGVzIjpbIkV2ZXJ5b25lIiwiQmFzaWMgT3BzIiwiQ1YgT3BzIiwiQ29kZXJzIE9wcyJdLCJuYW1lIjoiYXhlbC52bGFtaW5jayIsImVtYWlsIjoiYXhlbEBnb2xkZW5zZXRhbmFseXRpY3MuY29tIn0.kQEKSjUpABLtdgGqQ_PDPjRsggXTxMOeE91dufnprBG7pOC0yABOYJsQAqVZzdQ4jxrDhbA22N1KRKODB3r0x6DXg8j_PYPCe6DGOr7xhpN4StJnaYMHCNL7wPh3P1l262aJUlgkH3XDdDFZGqmWlmQvFEmVMfjKtx3teoisB7XKCMBd31rzfMvNC3QiEAf8SVAL8UCNbD7qrWQ2hl-DDvyt9oAGCqoybMeu6w1D2M99IxWeM-GXIRYYoVB-SvgwiQ6Ukg6b1zfNLUoL4o1ryHsfbMHccsgavZpK6Pbi58BaYqAUnhPSauTHb6jIvsI0EwA9dq8KkH5KmfYifEoFzw"
url = "https://api.goldensetanalytics.com"
headers = {
    'accept': '*/*',
    'accept-language': 'en-US,en;q=0.9',
    'authorization': f'Bearer {BEARER_TOKEN}',
    'content-length': '0',
    'origin': 'https://ops.goldensetanalytics.com',
    'priority': 'u=1, i',
    'sec-ch-ua': '"Chromium";v="128", "Not;A=Brand";v="24", "Google Chrome";v="128"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"macOS"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-site',
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36',
}


def send_video_request(video_id):
    request_url = url + f'/videos/cv/send?id={video_id}&env=dev&screenCoordinates=false&enable3d=true'
    response = requests.post(request_url, headers=headers)
    print(request_url, headers)
    logger.info(response.status_code)


def get_video_info(video_id):
    request_url = url + f'/videos/ready/{video_id}'
    response = requests.get(request_url, headers=headers)
    logger.info(response.status_code)
    return response.json()


def get_video_snapshot(video_id):
    request_url = url + f'/videos/{video_id}/snapshots'
    response = requests.get(request_url, headers=headers)
    logger.info(response.status_code)
    return response.json()


def download_report(video_id, output_path: pathlib.Path):
    #download_url = url + f'/videos/{video_id}/snapshots/download/report?fileName=report'
    download_url = url + f'/cv/download/{video_id}'
    response = requests.get(download_url, headers=headers)
    logger.info(response.status_code)
    response_data = response.json()
    response_data['match_id'] = video_id
    df = cv_json_to_df(response_data)
    df.to_parquet(str(output_path))
    logger.info(f"report version: {response_data['version']}")
    return response_data


def run_benchmark(matches_cv):
    for match, video_id in matches_cv.items():
        logger.info(f"Sending request for {match}")
        send_video_request(video_id)


def download_reports(matches_cv, output_dir: pathlib.Path):
    if True:
        for match, video_id in tqdm.tqdm(matches_cv.items(), desc="Downloading reports"):
            try:
                report_path = output_dir / f"{match}.parquet"
                report_path.parent.mkdir(parents=True, exist_ok=True)
                logger.info(f"Downloading report for {match}")
                download_report(video_id, report_path)
            except:
                print('match failed', video_id)
                continue
 



logging.basicConfig(level=logging.INFO)

cvs = [d['id'] for d in data]# + ['9923861b-fdaa-48ab-8de6-0322f45c07d5']
#cvs = [d['id'] for d in [data[4]]]

opelka = {k: k for k in cvs}

download_reports(opelka, pathlib.Path(f"{poi}_breeze52456"))

In [ ]:
# CV PDF GENERATOR UTILS

from srcimg3.pdf_generator.models.enums import (
    Fonts,
    Color,
    CourtSide,
    ReturnDirection,
    ServeDirection,
    Offset,
    ShotType,
    SurfaceCode,
)


from srcimg3.pdf_generator.visuals.templates import serve_location_template, rally_length_template, ColorPreset, good_returns_template, template_g, rally_ending_template





def create_serve_visual_win_cv(df, server, serve_number):
    serves = df[(df.shot == 1) & (df.serve == df.serve_number) & (df.impact_player == server)
           & (df.serve_number == serve_number)]
   
    ad_serves = serves[serves.deuce_or_ad == 'ad']
    deuce_serves = serves[serves.deuce_or_ad == 'deuce']

    deuce_serves_T = deuce_serves[deuce_serves.placement_gsa == 'T']
    deuce_serves_W = deuce_serves[deuce_serves.placement_gsa == 'Wide']
    deuce_serves_B = deuce_serves[deuce_serves.placement_gsa == 'Body']

    ad_serves_T = ad_serves[ad_serves.placement_gsa == 'T']
    ad_serves_W = ad_serves[ad_serves.placement_gsa == 'Wide']
    ad_serves_B = ad_serves[ad_serves.placement_gsa == 'Body']

    all_deuce_serves = len(deuce_serves_T) + len(deuce_serves_B) + len(deuce_serves_W)
    all_ad_serves = len(ad_serves_T) + len(ad_serves_B) + len(ad_serves_W)

    def get_ratio(x, y):
        if not y or np.isnan(y):
            return 0
        return int(round(100.0 * x / y))

    arrows_widths = [
        get_ratio(len(ad_serves_W), all_ad_serves),
        get_ratio(len(ad_serves_B), all_ad_serves),
        get_ratio(len(ad_serves_T), all_ad_serves),
        get_ratio(len(deuce_serves_T), all_deuce_serves),
        get_ratio(len(deuce_serves_B), all_deuce_serves),
        get_ratio(len(deuce_serves_W), all_deuce_serves),
    ]
    arrows_numbers= [f'{x}%' for x in arrows_widths]
    numbers = [len(ad_serves_W), len(ad_serves_B), len(ad_serves_T), len(deuce_serves_T), len(deuce_serves_B), len(deuce_serves_W)]
    pies_percentages = [get_ratio(len(x[x.impact_player == x.point_winner]), len(x)) for x in [ad_serves_W, ad_serves_B, ad_serves_T, deuce_serves_T, deuce_serves_B, deuce_serves_W]]

    court = serve_location_template(
        player_name=poi,
        opponent_name='Opponents',
        serve_no='1st' if serve_number == 1 else '2nd',
        arrows_widths= arrows_widths,
        arrows_numbers= arrows_numbers,
        surface= surfacecode,
        numbers= numbers,
        pies_percentages=pies_percentages,
        preset=ColorPreset.ORANGE,
        target_speed_unit = 'MPH'
    )
    return court

def create_serve_visual_win_cv(df, server, serve_number):
    serves = df[(df.shot == 1) & (df.serve == df.serve_number) & (df.impact_player == server)
           & (df.serve_number == serve_number)]
   
    ad_serves = serves[serves.deuce_or_ad == 'ad']
    deuce_serves = serves[serves.deuce_or_ad == 'deuce']

    deuce_serves_T = deuce_serves[deuce_serves.placement_gsa == 'T']
    deuce_serves_W = deuce_serves[deuce_serves.placement_gsa == 'Wide']
    deuce_serves_B = deuce_serves[deuce_serves.placement_gsa == 'Body']

    ad_serves_T = ad_serves[ad_serves.placement_gsa == 'T']
    ad_serves_W = ad_serves[ad_serves.placement_gsa == 'Wide']
    ad_serves_B = ad_serves[ad_serves.placement_gsa == 'Body']

    all_deuce_serves = len(deuce_serves_T) + len(deuce_serves_B) + len(deuce_serves_W)
    all_ad_serves = len(ad_serves_T) + len(ad_serves_B) + len(ad_serves_W)

    def get_ratio(x, y):
        if not y or np.isnan(y):
            return 0
        return int(round(100.0 * x / y))

    arrows_widths = [
        get_ratio(len(ad_serves_W), all_ad_serves),
        get_ratio(len(ad_serves_B), all_ad_serves),
        get_ratio(len(ad_serves_T), all_ad_serves),
        get_ratio(len(deuce_serves_T), all_deuce_serves),
        get_ratio(len(deuce_serves_B), all_deuce_serves),
        get_ratio(len(deuce_serves_W), all_deuce_serves),
    ]
    arrows_numbers= [f'{x}%' for x in arrows_widths]
    numbers = [len(ad_serves_W), len(ad_serves_B), len(ad_serves_T), len(deuce_serves_T), len(deuce_serves_B), len(deuce_serves_W)]
    pies_percentages = [get_ratio(len(x[x.impact_player == x.point_winner]), len(x)) for x in [ad_serves_W, ad_serves_B, ad_serves_T, deuce_serves_T, deuce_serves_B, deuce_serves_W]]

    court = serve_location_template(
        player_name=poi,
        opponent_name='Opponents',
        serve_no='1st' if serve_number == 1 else '2nd',
        arrows_widths= arrows_widths,
        arrows_numbers= arrows_numbers,
        surface= surfacecode,
        numbers= numbers,
        pies_percentages=pies_percentages,
        preset=ColorPreset.ORANGE,
        target_speed_unit = 'MPH'
    )
    return court

def create_return_visual_win_cv(df, server, serve_number):
    serves = df[(df.shot == 1) & (df.serve == df.serve_number) & (df.impact_player != server)
           & (df.serve_number == serve_number)]
   
    ad_serves = serves[serves.deuce_or_ad == 'ad']
    deuce_serves = serves[serves.deuce_or_ad == 'deuce']

    deuce_serves_T = deuce_serves[deuce_serves.placement_gsa == 'T']
    deuce_serves_W = deuce_serves[deuce_serves.placement_gsa == 'Wide']
    deuce_serves_B = deuce_serves[deuce_serves.placement_gsa == 'Body']

    ad_serves_T = ad_serves[ad_serves.placement_gsa == 'T']
    ad_serves_W = ad_serves[ad_serves.placement_gsa == 'Wide']
    ad_serves_B = ad_serves[ad_serves.placement_gsa == 'Body']

    all_deuce_serves = len(deuce_serves_T) + len(deuce_serves_B) + len(deuce_serves_W)
    all_ad_serves = len(ad_serves_T) + len(ad_serves_B) + len(ad_serves_W)

    def get_ratio(x, y):
        if not y or np.isnan(y):
            return 0
        return int(round(100.0 * x / y))

    arrows_widths = [
         get_ratio(len(deuce_serves_W), all_deuce_serves),
        get_ratio(len(deuce_serves_B), all_deuce_serves),
        get_ratio(len(deuce_serves_T), all_deuce_serves),
        get_ratio(len(ad_serves_T), all_ad_serves),
        get_ratio(len(ad_serves_B), all_ad_serves),
        get_ratio(len(ad_serves_W), all_ad_serves),
       
    ]
    arrows_numbers= [f'{x}%' for x in arrows_widths]
    numbers = [len(deuce_serves_W), len(deuce_serves_B), len(deuce_serves_T), len(ad_serves_T), len(ad_serves_B), len(ad_serves_W)]
    pies_percentages = [get_ratio(len(x[x.impact_player != x.point_winner]), len(x)) for x in [deuce_serves_W, deuce_serves_B, deuce_serves_T, ad_serves_T, ad_serves_B, ad_serves_W]]

    court = good_returns_template(
        player_name=poi,
        opponent_name='Opponents',
        serve_no='1st' if serve_number == 1 else '2nd',
        arrows_widths= arrows_widths,
        arrows_numbers= arrows_numbers,
        surface= surfacecode,
        numbers= numbers,
        pies_percentages=pies_percentages,
        preset=ColorPreset.ORANGE,
        target_speed_unit = 'MPH'
    )
    return court

In [ ]:
#poi = 'Mboko'

In [ ]:
poi

In [ ]:
# CV DATA MANIPULATION
import pandas as pd
from glob import glob
folder = f"{poi}_breeze52456"
data = []
for file in glob(folder + '/*'):
    data.append(pd.read_parquet(file))
df = pd.concat(data)

In [ ]:
def resolve_serve_attempts(df, point_cols=None):
    """Drop the rallies that were never played and number the rest 1st / 2nd serve.

    A point is a run of rallies, each opened by a serve row (shot_no == 1). Three
    kinds of rally never happened and must not consume a service attempt:

      * lets and phantom detections (outcome 'Let' / 'Extra');
      * a rally whose serve duplicates a later one in the same point — same
        placement, same landing spot, same speed — i.e. the CV wrote the same
        rally down twice;
      * anything before the last two rallies, since a point holds at most two
        serves.

    Returns `df` without those rows, plus three columns, each carrying the value
    of the rally the row belongs to (so returns and +1 shots inherit them):

      rally_no       1-based rally index within the point
      serve_attempt  attempt number (1 or 2)
      serve_outcome  'in' / 'out' for that attempt

    The frame's own attempt column (`serve_number` / `serve`) is overwritten with
    the corrected number so existing key/group code keeps working. Idempotent:
    re-running on an already-resolved frame changes nothing.
    """
    d = df.reset_index(drop=True).copy()
    shot = _pick(d, _C_SHOT)
    attempt_col = _pick(d, _C_ATTEMPT)
    is_serve = (pd.to_numeric(d[shot], errors="coerce") == 1) if shot else None

    if is_serve is None or not bool(is_serve.any()):
        # No serve rows to anchor the numbering on (e.g. a rally-only frame):
        # leave the caller's numbering alone rather than zeroing it out.
        d["rally_no"] = (pd.to_numeric(d[attempt_col], errors="coerce")
                         if attempt_col else 1)
        d["serve_attempt"] = d["rally_no"]
        d["serve_outcome"] = "in"
        return d

    drop_key = None
    if point_cols is None:
        d["_pt_key"] = _point_key(d)
        point_cols, drop_key = ["_pt_key"], "_pt_key"
    pt = d.groupby(point_cols, sort=False).ngroup()
    rally = is_serve.astype(int).groupby(pt).cumsum()

    oc = _pick(d, _C_OUTCOME)
    outcome = d[oc].astype(object) if oc else pd.Series(index=d.index, dtype=object)

    # 1. lets / phantom rallies: drop the whole rally its serve row opened
    junk = is_serve & outcome.isin(NON_ATTEMPT_OUTCOMES)
    drop = junk.groupby([pt, rally]).transform("max").astype(bool)

    # 2. duplicated serve rows, and anything before the last two attempts
    place = _pick(d, _C_PLACEMENT)
    bx, by, sp = _pick(d, _C_BOUNCE_X), _pick(d, _C_BOUNCE_Y), _pick(d, _C_SPEED)
    can_compare = all(c is not None for c in (place, bx, by, sp))
    if can_compare:
        x = pd.to_numeric(d[bx], errors="coerce")
        y = pd.to_numeric(d[by], errors="coerce")
        v = pd.to_numeric(d[sp], errors="coerce")

    dead = set()
    srv_idx = d.index[is_serve & ~drop]
    for _, idx in pd.Series(srv_idx).groupby(pt[srv_idx].values, sort=False):
        rallies = [(i, rally[i]) for i in idx]
        for (i, ri), (j, _) in zip(rallies, rallies[1:]):
            same_serve = can_compare and (
                d.at[i, place] == d.at[j, place]
                and pd.notna(x[i]) and pd.notna(x[j]) and abs(x[i] - x[j]) < DUPLICATE_BOUNCE_M
                and pd.notna(y[i]) and pd.notna(y[j]) and abs(y[i] - y[j]) < DUPLICATE_BOUNCE_M
                and pd.notna(v[i]) and pd.notna(v[j]) and abs(v[i] - v[j]) < DUPLICATE_SPEED
            )
            if same_serve:
                dead.add((pt[i], ri))       # the earlier of the pair is the copy
        # a point cannot hold more than two attempts — keep the last two
        alive = [(pt[i], ri) for i, ri in rallies if (pt[i], ri) not in dead]
        for key in alive[:-2]:
            dead.add(key)
    if dead:
        drop = drop | pd.Series(list(zip(pt, rally)), index=d.index).isin(dead)

    d = d[~drop].copy()
    pt, is_serve = pt[~drop], is_serve[~drop]
    rally_no = is_serve.astype(int).groupby(pt).cumsum()      # renumbered, contiguous
    attempt = rally_no.clip(upper=2)

    # in/out from the point structure: an attempt with another behind it was out;
    # the last attempt was in unless the point ended on a fault (double fault).
    played = attempt.groupby(pt).transform("max")
    if oc:
        faulted = d[oc].astype(object).eq("Fault")
    elif _pick(d, _C_IS_IN):
        faulted = pd.to_numeric(d[_pick(d, _C_IS_IN)], errors="coerce").eq(0)
    else:
        faulted = pd.Series(False, index=d.index)
    out = (attempt < played) | (is_serve & faulted)
    out = out.groupby([pt, attempt]).transform("max").astype(bool)

    d["rally_no"] = rally_no
    d["serve_attempt"] = attempt
    d["serve_outcome"] = np.where(out, "out", "in")
    for name in _C_ATTEMPT:                 # keep the frame's own column in sync
        if name in d.columns:
            d[name] = attempt
    if drop_key:
        d = d.drop(columns=[drop_key])
    return d.reset_index(drop=True)
def _pick(df, names):
    """First of `names` present in the frame, else None."""
    for n in names:
        if n in df.columns:
            return n
    return None
_C_SHOT = ("shot_no", "shot")
_C_ATTEMPT = ("serve_number", "serve")
_C_OUTCOME = ("outcome",)
_C_IS_IN = ("is_shot_in",)
_C_PLACEMENT = ("serve_direction", "placement_gsa", "placement")
_C_BOUNCE_X = ("REBOUND_X_abs", "bounce_x")
_C_BOUNCE_Y = ("REBOUND_Y_mirrored", "bounce_y")
_C_SPEED = ("SPEED", "speed")
def _point_key(d):
    """Point identifier, whether or not the frame carries point_id."""
    if "point_id" in d.columns:
        return d["point_id"].astype(str)
    for cols in (("match_id", "set_no", "game_no", "point_no"),
                 ("match_id", "set", "game", "point")):
        present = [c for c in cols if c in d.columns]
        if len(present) >= 2:
            key = d[present[0]].astype(str)
            for c in present[1:]:
                key = key + "_" + d[c].astype(str)
            return key
    return pd.Series("0", index=d.index)
NON_ATTEMPT_OUTCOMES = ("Let", "Extra")   # rallies that never consumed an attempt
DUPLICATE_BOUNCE_M = 0.3     # two serve rows landing within this (m) of each other...
DUPLICATE_SPEED = 10.0       # ...at this similar a speed, same placement = one serve
                             # (SPEED unit-agnostic: the gate is a loose sanity check)
from cv_adapter import add_he_features
df = add_he_features(df)
df = resolve_serve_attempts(df)